In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, LabelEncoder,StandardScaler
import seaborn as sns
from sklearn.model_selection import KFold,train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Download latest version
#path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")

In [ ]:
# Task 2: Write your code here:
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# Delivery_Time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
from matplotlib import color_sequences
# Task 1: Write your code here:
df_food=df_food.drop(columns=['Order_ID'])
df_food.head()

In [ ]:
# Task 2: Write your code here:
df_food.isnull().sum()

In [ ]:
# Analyze missing values
missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)
#since there is small number of nan i will fill them delivry time will take the mean

In [ ]:
# Task 2: Write your code here:
for col in ['Weather', 'Time_of_Day', 'Traffic_Level', 'Courier_Experience_yrs']:
    df_food[col] = df_food[col].fillna('unknown')

df_food['Delivery_Time'] = df_food['Delivery_Time'].fillna(df_food['Delivery_Time'].mean)
print("Missing values remaining:", df_food.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food)

In [ ]:
# Task 4: Write your code here: aclot of obj :>
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Vehicle_Type','Time_of_Day','Courier_Experience_yrs',]
for col in categorical_cols:
    le = LabelEncoder()
    df_food[col] = le.fit_transform(df_food[col].astype(str))

df_food.head()

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
df_food_scaled=scaler.fit_transform(df_food.drop(columns=['Delivery_Time']))
df_food.head()


In [ ]:
df_food.info()

In [ ]:
# Task 1: Write your code here:
x=df_food_scaled
y=df_food['Delivery_Time']


In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model=  {"Random Forest": RandomForestRegressor(
      n_estimators=320,  # Number of trees
      )
  }

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(x):
    X_fold_train, X_fold_val = x.iloc[train_idx], x.iloc[val_idx]
    y_fold_train, y_fold_val = x.iloc[train_idx], x.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")








In [ ]:
# Task 1: Write your code here:
# Retrieve CatBoost feature importances and sort them
catboost_model = model["CatBoost Classifier"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('ranom forest Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: